# 02 â€” Chunk & Embed
## Databricks Expert Agent Project

**What this notebook does:**
1. Reads raw docs from `chatbot.rag_chatbot.raw_docs`
2. Splits each page into overlapping chunks (~1,000 chars, 200 overlap)
3. Embeds each chunk using Databricks Foundation Model API (gte-large-en)
4. Writes chunks + embeddings to `chatbot.rag_chatbot.doc_chunks`

**Output:** A Delta table with one row per chunk, including a 1,024-dimension embedding vector â€” ready for Vector Search indexing in Notebook 3.

In [0]:
# â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CATALOG      = "chatbot"
SCHEMA       = "rag_chatbot"
RAW_TABLE    = f"{CATALOG}.{SCHEMA}.raw_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.doc_chunks"

CHUNK_SIZE    = 1000   # characters per chunk
CHUNK_OVERLAP = 200    # overlap between consecutive chunks
EMBED_MODEL   = "databricks-gte-large-en"   # Foundation Model API endpoint
EMBED_BATCH   = 25     # max inputs per API call (model limit)

print(f"Source  : {RAW_TABLE}")
print(f"Target  : {CHUNKS_TABLE}")
print(f"Chunks  : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"Model   : {EMBED_MODEL}")

In [0]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    if not text or len(text) < 100:
        return []
    chunks = []
    step = chunk_size - overlap
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if len(chunk.strip()) > 100:
            chunks.append(chunk.strip())
        start += step
    return chunks

In [0]:
from pyspark.sql.functions import udf, posexplode, concat, lit, col, when
from pyspark.sql.types import ArrayType, StringType

# Wrap chunker as a UDF that returns an array of strings
chunk_udf = udf(lambda text: chunk_text(text) if text else [], ArrayType(StringType()))

# Read raw docs
raw_df = spark.table(RAW_TABLE)

# Add source metadata if old raw_docs table does not have it
if "source" not in raw_df.columns:
    raw_df = raw_df.withColumn(
        "source",
        when(col("url").startswith("internal_wiki://"), lit("internal_wiki"))
        .otherwise(lit("microsoft_learn_azure_databricks"))
    )

if "cloud" not in raw_df.columns:
    raw_df = raw_df.withColumn("cloud", lit("azure"))

# Apply chunker â€” produces an array of chunk strings per row
# posexplode turns array into multiple rows and gives us position (pos) of each chunk
chunks_df = (
    raw_df
    .select(
        "url",
        "title",
        "scraped_date",
        "source",
        "cloud",
        chunk_udf("content").alias("chunks")
    )
    .select(
        "url",
        "title",
        "scraped_date",
        "source",
        "cloud",
        posexplode("chunks").alias("chunk_index", "chunk_text")
    )
    .filter("length(chunk_text) > 100")
)

# Add a source_type to help the app rank internal runbooks higher later
chunks_df = chunks_df.withColumn(
    "source_type",
    when(col("source") == "internal_wiki", lit("internal"))
    .otherwise(lit("official_docs"))
)

# Add a unique chunk_id â€” url + position so it's human readable and unique
chunks_df = chunks_df.withColumn(
    "chunk_id",
    concat(col("url"), lit("::chunk::"), col("chunk_index").cast("string"))
)

print(f"Total chunks : {chunks_df.count():,}")
print(f"Avg per page : {chunks_df.count() / raw_df.count():.1f}")

display(chunks_df.select(
    "chunk_id",
    "url",
    "title",
    "source",
    "source_type",
    "cloud",
    "chunk_index",
    "chunk_text"
).limit(5))

In [0]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")

try:
    test_response = deploy_client.predict(
        endpoint=EMBED_MODEL,
        inputs={"input": ["What is Delta Lake and how does it work?"]}
    )
    embedding = test_response.data[0]["embedding"]
    print(f"âœ“ API call succeeded")
    print(f"âœ“ Embedding dimensions : {len(embedding)}")
    print(f"âœ“ First 5 values       : {embedding[:5]}")
except Exception as e:
    raise RuntimeError(f"Embedding endpoint test failed for '{EMBED_MODEL}': {e}")

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, FloatType
import mlflow.deployments
import time

EMBED_MODEL  = "databricks-gte-large-en"
EMBED_BATCH  = 25

@pandas_udf(ArrayType(FloatType()))
def embed_udf(texts: pd.Series) -> pd.Series:
    """
    Pandas UDF â€” receives a batch of text strings, 
    calls the embedding API in sub-batches of 25,
    returns a list of 1024-float vectors.
    """
    client = mlflow.deployments.get_deploy_client("databricks")
    results = []
    batch = texts.tolist()

    # Split into sub-batches of EMBED_BATCH size
    for i in range(0, len(batch), EMBED_BATCH):
        sub_batch = batch[i : i + EMBED_BATCH]
        try:
            resp = client.predict(
                endpoint=EMBED_MODEL,
                inputs={"input": sub_batch}
            )
            for item in resp.data:
                results.append(item["embedding"])
        except Exception as e:
            # On failure, append None vectors so we don't lose the row
            for _ in sub_batch:
                results.append(None)
            print(f"Embedding batch {i} failed: {e}")
        time.sleep(0.1)   # brief pause to avoid rate limit spikes

    return pd.Series(results)

print("âœ“ Embed UDF defined â€” ready to run")

In [0]:
print(f"Embedding {chunks_df.count():,} chunks â€” expect 15-25 minutes...\n")

# Apply the embedding UDF to add a vector column
embedded_df = chunks_df.withColumn("embedding", embed_udf(col("chunk_text")))

# Drop any rows where embedding failed
embedded_df = embedded_df.filter(col("embedding").isNotNull())

# Write to Delta
(
    embedded_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHUNKS_TABLE)
)

final_count = spark.table(CHUNKS_TABLE).count()
print(f"âœ“ Written to : {CHUNKS_TABLE}")
print(f"âœ“ Row count  : {final_count:,}")

In [0]:
from pyspark.sql.functions import col

df = spark.table(CHUNKS_TABLE)
df.printSchema()

print(f"\nTotal chunks     : {df.count():,}")
print(f"Null embeddings  : {df.filter(col('embedding').isNull()).count()}")
print(f"Embedding length : {len(df.select('embedding').first()[0])}")

display(df.select("title", "chunk_index", "chunk_text").limit(3))